In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Getting Started with OpenAI GPT Models
<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/vertex-ai-samples/blob/main/notebooks/official/generative_ai/openai_gpt_intro.ipynb">
      <img src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fvertex-ai-samples%2Fmain%2Fnotebooks%2Fofficial%2Fgenerative_ai%2Fopenai_gpt_intro.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/notebooks/deploy-notebook?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/vertex-ai-samples/main/notebooks/official/generative_ai/openai_gpt_intro.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/vertexai/v1/32px.svg" alt="Vertex AI logo"><br> Open in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/vertex-ai-samples/blob/main/notebooks/official/generative_ai/openai_gpt_intro.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

## Overview

### OpenAI on Gemini platform

OpenAI GPT models are available as managed, serverless Model-as-a-Service (MaaS) endpoints on the **Gemini platform** through Google Cloud. This integration combines OpenAI's frontier model capabilities with Google Cloud's enterprise-grade security, scalability, and operational management:

* **Identity and Access Management:** Authenticate using existing **Google Cloud IAM** roles and service accounts, eliminating the need to provision, rotate, or manage separate API keys.
* **OpenAI API Compatibility:** The API surface is compatible with the **OpenAI Responses API** (`publishers/openai/v1/responses`), allowing existing codebases, SDKs, and tooling built against the OpenAI Responses API to work with updates to the base URL, model ID, and Google Cloud authentication.
* **Unified Billing:** All model consumption appears directly on your **unified Google Cloud invoice** and is eligible for Google Cloud commitment drawdown, subject to standard Marketplace drawdown caps.
* **Security and Enterprise Data Privacy:** All execution takes place within **Google Cloud's security perimeter**. Your prompts, context, and generated responses are strictly protected and **never used to train underlying models**.

### Available OpenAI GPT models

Note: For the latest list of available GPT models on Google Cloud, please refer to the [Partner Models Overview](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/partner-models/use-partner-models) 

#### GPT-6.0 Luna (`gpt-6-luna`)
OpenAI's high-efficiency reasoning model in the GPT-6.0 family, ideal for high-throughput agentic and interactive workloads.

#### GPT-6.1 Sol (`gpt-6.1-sol`)
Next-generation upgrade to GPT-6.0 Sol with enhanced coding, tool calling, and long-horizon agentic capabilities.

#### GPT-6.0 Astra (`gpt-6-astra`)
OpenAI's frontier model in the GPT-6.0 family for complex reasoning and deep problem solving.

### Prerequisites

Before you begin, ensure you have the following:

* **Google Cloud Project:** A [Google Cloud Project](https://cloud.google.com/resource-manager/docs/creating-managing-projects) with billing enabled and allowlisted for OpenAI models.
* **Enabled APIs:** [Gemini Enterprise Agent Platform / Vertex AI API (`aiplatform.googleapis.com`)](https://console.cloud.google.com/apis/enableflow?apiid=aiplatform.googleapis.com).
* **Model Garden Enablement:** OpenAI GPT models enabled in Model Garden.
* **IAM Permissions:** [Agent Platform User (`roles/aiplatform.user`)](https://cloud.google.com/iam/docs/roles-permissions/aiplatform) on the project for the user or service account making requests.
* **Local / Notebook Environment:** Python 3.10+ and the [gcloud CLI](https://cloud.google.com/sdk/docs/install-sdk) with [Application Default Credentials (ADC)](https://cloud.google.com/docs/authentication/set-up-adc-local-dev-environment) configured.

## Objective

This notebook shows how to:
1. Enable OpenAI GPT models in Model Garden and configure `vertexai.allowedModels` Organization Policies (if required).
2. Call OpenAI GPT models using **cURL** against the Agent Platform / Vertex AI REST API (unary and streaming).
3. Call OpenAI GPT models using the **OpenAI Python SDK** (`openai`) with Google Cloud authentication (unary, streaming, multimodal vision, and environment variable setup).
4. Configure and run the **OpenAI Codex CLI** against the Google Cloud endpoint.
5. Configure and run the **OpenAI Codex Desktop App** against the Google Cloud endpoint.

## Get Started

### Install required packages

In [ ]:
%pip install -U -q openai google-auth requests httpx ipywidgets

### Authenticate your notebook environment

If you are running this notebook on Google Colab, run the cell below to authenticate your environment.

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

If you are running locally or in a shell environment, configure Application Default Credentials (ADC) by running:
```bash
gcloud auth application-default login
```

#### Select OpenAI model

Throughout this notebook, we default to **GPT-6.0 Luna** (`gpt-6-luna`). You can also select `gpt-6-luna`, `gpt-6-astra`, or `gpt-6.1-sol`.

In [ ]:
MODEL = "gpt-6-luna"  # @param ["gpt-6-sol", "gpt-6-luna", "gpt-6-astra", "gpt-6.1-sol"]
available_regions = ["global", "us"]

#### Select a location

Two endpoint scopes are available:
* **`global` *(recommended)*:** `aiplatform.googleapis.com`
* **`us` *(US multi-region)*:** `aiplatform.us.rep.googleapis.com`

Use the `global` endpoint unless you have a specific data residency requirement that calls for the `us` multi-region endpoint.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

dropdown = widgets.Dropdown(
    options=available_regions,
    value="global",
    description="Select a location:",
    font_weight="bold",
    style={"description_width": "initial"},
)


def dropdown_eventhandler(change):
    global LOCATION
    if change["type"] == "change" and change["name"] == "value":
        LOCATION = change.new
        print("Selected:", change.new)


LOCATION = dropdown.value
dropdown.observe(dropdown_eventhandler, names="value")
display(dropdown)

#### Set Google Cloud project and endpoint information

To get started, you must have an existing Google Cloud project and [enable the Gemini Enterprise Agent Platform (Vertex AI) API](https://console.cloud.google.com/apis/enableflow?apiid=aiplatform.googleapis.com). Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [ ]:
import os
import subprocess

# fmt: off
PROJECT_ID = "[your-project-id]"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
# fmt: on
if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "").strip()
if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = subprocess.check_output(
        ["gcloud", "config", "get-value", "project"], text=True
    ).strip()
if not PROJECT_ID or PROJECT_ID in ("[your-project-id]", "(unset)"):
    raise ValueError(
        "Please set PROJECT_ID or run `gcloud config set project YOUR_PROJECT_ID`."
    )

ENDPOINT = (
    "aiplatform.googleapis.com"
    if LOCATION == "global"
    else f"aiplatform.{LOCATION}.rep.googleapis.com"
)
BASE_URL = f"https://{ENDPOINT}/v1/projects/{PROJECT_ID}/locations/{LOCATION}/publishers/openai/v1"

### First Steps: Enable the Model & Configure Organization Policy

#### 1. Enable the model in Model Garden

You can enable OpenAI models either in the Cloud Console Model Garden UI (by opening the OpenAI model card and completing the enablement and agreement steps) or programmatically via the API.

The models are served on `global` and `us` endpoints for inference, and are enabled at the regional level (e.g. `us-east1`).

In [ ]:
PUBLISHER = "openai"
ENABLE_ENDPOINT = "us-east1-aiplatform.googleapis.com"
PARENT = f"projects/{PROJECT_ID}"
NAME = f"publishers/{PUBLISHER}/models/{MODEL}"

!curl -X POST \
  -H "Authorization: Bearer $(gcloud auth application-default print-access-token)" \
  -H "x-goog-user-project: {PROJECT_ID}" \
  -H "Content-Type: application/json; charset=utf-8" \
  -d '{{}}' \
  "https://{ENABLE_ENDPOINT}/v1beta1/{PARENT}/{NAME}:enableModel" 

#### 2. Allowlist the model in Organization Policy (if required)

If your organization restricts model access using the [`vertexai.allowedModels`](https://cloud.google.com/gemini-enterprise-agent-platform/models/control-model-access#allow_a_set_of_models_and_deny_all_other_models) organization policy constraint, include the required model slugs in your allowlist:

* `publishers/openai/models/gpt-6-astra:predict`
* `publishers/openai/models/gpt-6.1-sol:predict`
* `publishers/openai/models/gpt-6-luna:predict`

## Making Model Requests Using cURL (Vertex AI / Agent Platform REST API)

#### Import required libraries

In [ ]:
import base64
import json  # noqa: F401

import httpx
from IPython.display import Image

### Text generation

#### Unary call (No streaming)

Sends a synchronous `POST` request to the OpenAI Responses API endpoint (`publishers/openai/v1/responses`). A successful request returns a completed response object with output text.

In [ ]:
PAYLOAD = {
    "model": MODEL,
    "input": "Who are you?",
    "store": False,
}

request = json.dumps(PAYLOAD)
!curl --fail-with-body -X POST \
  -H "Content-Type: application/json" \
  -H "Authorization: Bearer $(gcloud auth application-default print-access-token)" \
  "https://{ENDPOINT}/v1/projects/{PROJECT_ID}/locations/{LOCATION}/publishers/openai/v1/responses" \
  -d '{request}'

#### Streaming call (With streaming)

Sends a streaming `POST` request with `"stream": True`. Streaming requests return server-sent events (SSE), including `response.output_text.delta` events followed by `response.completed`.

In [ ]:
PAYLOAD = {
    "model": MODEL,
    "input": "Who are you?",
    "store": False,
    "stream": True,
}

request = json.dumps(PAYLOAD)
!curl --fail-with-body --no-buffer -X POST \
  -H "Content-Type: application/json" \
  -H "Authorization: Bearer $(gcloud auth application-default print-access-token)" \
  "https://{ENDPOINT}/v1/projects/{PROJECT_ID}/locations/{LOCATION}/publishers/openai/v1/responses" \
  -d '{request}'

#### Calling the US multi-region endpoint

If you have a US data residency requirement, target `LOCATION = "us"` and host `aiplatform.us.rep.googleapis.com`:

In [ ]:
US_LOCATION = "us"
US_ENDPOINT = f"aiplatform.{US_LOCATION}.rep.googleapis.com"

PAYLOAD = {
    "model": MODEL,
    "input": "Who are you?",
    "store": False,
}

request = json.dumps(PAYLOAD)
!curl --fail-with-body -X POST \
  -H "Content-Type: application/json" \
  -H "Authorization: Bearer $(gcloud auth application-default print-access-token)" \
  "https://{US_ENDPOINT}/v1/projects/{PROJECT_ID}/locations/{US_LOCATION}/publishers/openai/v1/responses" \
  -d '{request}'

## Using the OpenAI Python SDK

The [OpenAI Python library](https://github.com/openai/openai-python) (`openai`) provides convenient access to OpenAI models on Google Cloud from any Python 3.10+ application.

### Option A: Configure the OpenAI client with Google Auth (Recommended for Notebooks / Colab)

Use `google.auth` to obtain Application Default Credentials (ADC) and pass the bearer token and Vertex AI / Agent Platform `base_url` to `OpenAI(...)`.

> **Note:** ADC access tokens expire in ~1 hour. If you encounter `401` errors in long-running jobs, call `creds.refresh(google.auth.transport.requests.Request())` again and recreate the client (or set `client.api_key = creds.token`).

In [ ]:
import google.auth
import google.auth.transport.requests
from openai import OpenAI

ENDPOINT = (
    "aiplatform.googleapis.com"
    if LOCATION == "global"
    else f"aiplatform.{LOCATION}.rep.googleapis.com"
)

creds, _ = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)
creds.refresh(google.auth.transport.requests.Request())

client = OpenAI(
    base_url=f"https://{ENDPOINT}/v1/projects/{PROJECT_ID}/locations/{LOCATION}/publishers/openai/v1",
    api_key=creds.token,
)

### Text generation

#### Unary call (Without streaming)

Sends a synchronous request via `client.responses.create` and prints the response.

In [ ]:
response = client.responses.create(
    model=MODEL,
    input="Who are you?",
    store=False,
)
print(response.model_dump_json(indent=2))

#### Streaming call (With streaming)

Sends a streaming request via `client.responses.create(..., stream=True)` and prints each event as it arrives.

In [ ]:
stream = client.responses.create(
    model=MODEL,
    input="Who are you?",
    stream=True,
    store=False,
)
for event in stream:
    print(event)

#### Controlling reasoning effort

You can configure the model's reasoning depth using the `reasoning` parameter. Supported `effort` levels for `gpt-6-sol` include:
* `low`: Fast responses with lighter reasoning
* `medium` *(default)*: Balances speed and reasoning depth for everyday tasks
* `high`: Greater reasoning depth for complex problems
* `xhigh`: Extra high reasoning depth for complex problems
* `max`: Maximum reasoning depth for the hardest problems
* `ultra`: Maximum reasoning with automatic task delegation

In [ ]:
response = client.responses.create(
    model=MODEL,
    input="Design a fault-tolerant retry strategy with exponential backoff and jitter for HTTP 429 and 5xx responses.",
    reasoning={"effort": "medium"},
    store=False,
)
print(response.output_text)

### Vision (Multimodal input)

#### Encode and preview image

We fetch a sample image from the public `cloud-samples-data` Cloud Storage bucket using `httpx`, encode it as base64, and preview it inline.

In [ ]:
image_url = (
    "https://storage.googleapis.com/cloud-samples-data/generative-ai/image/scones.jpg"
)
image_media_type = "image/jpeg"
image_bytes = httpx.get(image_url).content
image_b64 = base64.b64encode(image_bytes).decode("utf-8")

Image(image_bytes, width=300, height=200)

#### Multimodal call with image and text

Sends a base64-encoded image (`input_image`) alongside a text prompt (`input_text`) using the Responses API.

In [ ]:
response = client.responses.create(
    model=MODEL,
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": "Describe this image."},
                {
                    "type": "input_image",
                    "image_url": f"data:{image_media_type};base64,{image_b64}",
                },
            ],
        }
    ],
    store=False,
)
print(response.output_text)

### Option B: Configure via Environment Variables (Recommended for Shell / Local Scripts)

In shell scripts or local applications, you can export `OPENAI_BASE_URL` and `OPENAI_API_KEY` so `OpenAI()` picks them up automatically without code changes:

```bash
export PROJECT_ID="your-project-id"
export LOCATION="global"
export OPENAI_BASE_URL="https://aiplatform.googleapis.com/v1/projects/${PROJECT_ID}/locations/${LOCATION}/publishers/openai/v1"
export OPENAI_API_KEY="$(gcloud auth application-default print-access-token)"
```

> **Note:** Run the `export` commands and Python script in the same shell environment. When the token expires, re-run the `OPENAI_API_KEY` export and restart the script, or update the running client's `api_key` with a refreshed token.

In [ ]:
import os

from openai import OpenAI

os.environ["OPENAI_BASE_URL"] = (
    f"https://{ENDPOINT}/v1/projects/{PROJECT_ID}/locations/{LOCATION}/publishers/openai/v1"
)
os.environ["OPENAI_API_KEY"] = creds.token

env_client = (
    OpenAI()
)  # Picks up OPENAI_BASE_URL and OPENAI_API_KEY from environment variables
response = env_client.responses.create(
    model=MODEL,
    input="Who are you?",
    store=False,
)
print(response.output_text)

## Making Model Requests Using Codex CLI and Codex Desktop App

You can use the **OpenAI Codex CLI** (`0.162.0` or later) against the Google Cloud endpoint using a local model catalog and a dedicated `CODEX_HOME` directory.

### 1. Install Codex CLI (`0.162.0` or later)

Install [Node.js](https://nodejs.org/en/download) first, then install the Codex CLI release via `npm`:

```bash
# Install using npm
npm install -g @openai/codex

# Check the version. Should be codex-cli 0.162.0 or greater
codex --version

```

### 2. Create a new `config.toml` file for a new codex profile or update your existing `config.toml`. Below, we are creating a new `config.toml` file.

Run the cell below to generate a dedicated `codex_home/` directory containing `config.toml` pre-configured with your `PROJECT_ID`, `LOCATION`, and `MODEL` (e.g. `gpt-6-luna`).

In [ ]:
import pathlib

codex_home = pathlib.Path("./codex_home")
codex_home.mkdir(parents=True, exist_ok=True)

config_toml_path = codex_home / "config.toml"
config_toml_content = f"""# Use a dedicated CODEX_HOME for testing.
model_provider = "gcp"
model = "{MODEL}"
model_reasoning_summary = "none"
web_search = "disabled"

[model_providers.gcp]
name = "Google Cloud Vertex AI"
base_url = "https://{ENDPOINT}/v1/projects/{PROJECT_ID}/locations/{LOCATION}/publishers/openai/v1"
wire_api = "responses"
requires_openai_auth = false
supports_websockets = false
supports_standalone_web_search = false

[model_providers.gcp.auth]
command = "gcloud"
args = ["auth", "application-default", "print-access-token"]
timeout_ms = 30000
refresh_interval_ms = 3000000

[model_providers.gcp.capabilities]
external_web_access = false
remote_compaction = "v2"

[features]
image_generation = false
apps = false
plugins = true
multi_agent_v2 = false

[agents]
enabled = false
"""
config_toml_path.write_text(config_toml_content)
print(f"Wrote {config_toml_path}")

### 3. Start Codex CLI

In your terminal, launch `codex` pointing `CODEX_HOME` to the folder containing `config.toml` (set `CODEX_HOME` to the directory path, not the `config.toml` file path):

```bash
CODEX_HOME="/absolute/path/of/folder/where/config.toml/is" codex
```
### 4. Download and Install the latest version of the App
Download the latest version of the [Codex Desktop App](https://openai.com/codex/)

### 5. Launch the desktop app using your custom profile as below. This will allow you to use the Codex Desktop app with GPT models on Google Cloud
```bash
GCP_CODEX_HOME="/absolute/path/of/folder/where/config.toml/is"
/usr/bin/open -n \
  --env "CODEX_HOME=$GCP_CODEX_HOME" \
  --env "CODEX_ELECTRON_USER_DATA_PATH=$GCP_CODEX_HOME/desktop-profile" \
  "/Applications/ChatGPT.app" \
  --args "--user-data-dir=$GCP_CODEX_HOME/desktop-profile"
```

## Error Handling & Operational Reference

### Error handling

Build exponential backoff with jitter on `429` and `5xx` responses (capped at a handful of attempts) into production workloads.

| Status | Meaning | Recommended action |
| :---: | :--- | :--- |
| `401` | Token expired or invalid | Refresh the access token and retry |
| `403` | Identity lacks required permissions, or the API, project, model, or caller is not enabled | Verify IAM bindings, API enablement, project allowlisting, and that the model is enabled on the project |
| `404` | Endpoint path, location, or model identifier incorrect | Check the URL structure and location segment, and verify that the model ID is correct (e.g. `gpt-6-astra`, `gpt-6-luna`, or `gpt-6.1-sol`) |
| `429` | Rate, quota, or available capacity limit reached | Back off exponentially and retry; contact your account team if sustained |
| `5xx` | Transient backend condition | Retry with exponential backoff |